# HIT-ASR — main results on Earnings-22

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huseyin-karaca/hit-asr/blob/main/notebooks/main_earnings22.ipynb)

This notebook reproduces the main experiment of the paper on Earnings-22. Every model's hyperparameters are selected on a
25 % hold-out of the corpus; every model is then trained and tested on the same ten folds of a 5×2 cross-validation
of the remaining clips, and HIT-ASR is compared with every other system on those folds.

The configuration is fixed in `hitasr.configs.MAIN["earnings22"]` and the code is in `hitasr.experiment`. The data and the
stored searches and folds are read from the Hugging Face dataset [`huseyin-karaca/hit-asr`](https://huggingface.co/datasets/huseyin-karaca/hit-asr); nothing is written to it.

| Level | What runs | Needs |
|---|---|---|
| **1** | every table, from the searches and folds stored on the Hub | a CPU, minutes |
| **2** | every fold retrained from the published labels and frames (`RETUNE = True`: the search as well) | a GPU, hours |
| **3** | level 2 on labels and frames rebuilt from the audio with `extract` (set `HITASR_HUB` to that repository before the imports) | a GPU |

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("hitasr") is None:     # Colab: install the package (a local or Docker run already has it)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "hit-asr[paper] @ git+https://github.com/huseyin-karaca/hit-asr.git@main"], check=True)

In [2]:
from hitasr.configs import MAIN
from hitasr.experiment import MainExperiment

LEVEL = 1          # 1, 2 or 3 (see above)
RETUNE = False     # levels 2 and 3: run the hyperparameter search again
DEVICE = "cuda"    # levels 2 and 3 train on it

exp = MainExperiment(MAIN["earnings22"], level=LEVEL, retune=RETUNE, device=DEVICE)

Earnings-22: level 1 — search read from the Hub, folds read from the Hub
experts  Cohere-Transcribe, Kyutai-STT-2.6b, Parakeet-TDT-0.6b-v3
arms     HIT-ASR, MLP-pool, ADASTT, ROVER (confidence-weighted), CN-MBR, MLP-pool (hard CE)


## 1. Hyperparameter spaces (Table 2)

The free hyperparameters of every model and their choices; every other parameter is fixed
(`hitasr.configs.SPACES`).

In [3]:
exp.spaces()

Table 2 — hyperparameter spaces
10 random draws per arm; parameters not listed are fixed (hitasr.configs.SPACES)
  [HIT-ASR  (16 free of 20)]
      parameter                                     choices
        d_model                                  {128, 256}
        n_heads                                      {4, 8}
  stage1_layers                                   {1, 2, 3}
  stage2_layers                                      {1, 2}
        ffn_dim                            {256, 512, 1024}
        dropout                                {0.05, 0.15}
         fusion           {bridge, self_attn, concat, mean}
   share_stage1                               {True, False}
     input_norm                  {none, scale, standardize}
             lr                                    {0.0001}
   weight_decay                          {0.001, 0.01, 0.1}
     batch_size                             {8, 16, 32, 64}
     max_frames                           {500, 1000, 2000}
       patience                                     {5, 10}
    loss_preset {hard_ce_only, soft_ce_only, wer+hard, all}
label_smoothing                                  {0.0, 0.1}
  [MLP-pool  (8 free of 12)]
      parameter                        choices
       d_hidden         {256, 512, 1024, 2048}
       n_layers                      {1, 2, 3}
        dropout              {0.05, 0.15, 0.3}
             lr {3e-05, 0.0001, 0.0003, 0.001}
   weight_decay             {0.001, 0.01, 0.1}
     batch_size             {32, 64, 128, 256}
       patience                        {5, 10}
label_smoothing                     {0.0, 0.1}
  [ADASTT  (7 free of 8)]
       parameter                choices
    n_estimators       {100, 400, 1000}
   learning_rate {0.01, 0.03, 0.1, 0.3}
       max_depth              {3, 5, 8}
       subsample        {0.6, 0.8, 1.0}
colsample_bytree   {0.1, 0.3, 0.6, 1.0}
min_child_weight       {1.0, 4.0, 16.0}
      reg_lambda       {0.3, 3.0, 30.0}
  [ROVER (confidence-weighted)  (6 free of 6)]
     parameter                          choices
         alpha                            {0.0}
   conf_source               {cn, probe, blend}
     weighting {uniform, inverse_wer, log_odds}
weight_by_conf                    {False, True}
     conf_temp                            {2.0}
  null_penalty                      {0.25, 0.5}
  [CN-MBR  (7 free of 7)]
  parameter                          choices
     decode                      {slot, hyp}
 candidates              {members+consensus}
      prior {uniform, inverse_wer, log_odds}
temperature                           {10.0}
 null_prior                      {0.05, 0.2}
   use_conf                    {False, True}
conf_source                          {probe}
  [MLP-pool (hard CE)  (8 free of 12)]
      parameter                        choices
       d_hidden         {256, 512, 1024, 2048}
       n_layers                      {1, 2, 3}
        dropout              {0.05, 0.15, 0.3}
             lr {3e-05, 0.0001, 0.0003, 0.001}
   weight_decay             {0.001, 0.01, 0.1}
     batch_size             {32, 64, 128, 256}
       patience                        {5, 10}
label_smoothing                     {0.0, 0.1}

## 2. Data

The three experts' transcripts and word-error counters for every clip (at levels 2 and 3, also their frame-level
encoder states). The stored counters are recomputed from the stored transcripts as a check.

In [4]:
exp.load()

earnings22: 12,088 utterances x 3 experts (0 adopted from fastt) in 2.2s
100.0% of 1,200 (utterance, model) counters reproduce exactly


RouterStore(earnings22, cohere_transcribe+kyutai_stt_2_6b+parakeet_tdt_0_6b_v3, 12,088 rows, frames closed)

## 3. Hyperparameter search

A random 25 % of the clips is held out for the search. For every model, ten configurations are drawn at random from
its space (Table 2), each trained on 70 % of the hold-out and scored on the remaining 30 %; the configuration with the
lowest corpus WER is kept. The other 75 % of the clips are never seen by the search.

In [5]:
exp.search()

partition,n_utts,share
tune,3022,0.250
eval,9066,0.750
fit,2115,0.175
val,907,0.075


  on the tuning-score rows: bsm(kyutai_stt_2_6b) 0.1433   oracle 0.1014   gap 0.0419
  the 9,066 evaluation rows are never seen by a trial
  HIT-ASR 10 trials; MLP-pool 10 trials; ADASTT 10 trials; ROVER (confidence-weighted) 10 trials; CN-MBR 10 trials; MLP-pool (hard CE) 10 trials — read from the Hub


HPTRun('hpt_earnings22_c1f77e3f', 6 arm(s), best hit_asr 0.1270, eval on 9,066 rows)

In [6]:
exp.dataset()

statistic,earnings22
Total clips,12088
Hyperparameter hold-out,3022
Evaluation rows (5x2),9066
Train size (per fold),4533
Test size (per fold),4533
Hours,25.8884
Mean clip length (s),7.7100
Reference words,217118


In [7]:
exp.selected()

label,trials,hold-out WER,selected configuration
HIT-ASR,10,0.1270,"{""batch_size"": 64, ""d_model"": 256, ""dropout"": 0.15, ""ffn_dim"": 512, ""fusion"": ""mean"", ""input_norm"": ""none"", ""label_smoothing"": 0.1, ""loss_preset"": ""wer+hard"", ""lr"": 0.0001, ""max_frames"": 500, ""max_restarts"": 0, ""n_heads"": 4, ""n_seeds"": 3, ""patience"": 5, ""pooled_skip"": false, ""share_stage1"": false, ""stage1_layers"": 2, ""stage2_layers"": 2, ""weight_decay"": 0.01, ""word_weighted"": false}"
MLP-pool,10,0.1296,"{""batch_size"": 256, ""d_hidden"": 1024, ""dropout"": 0.15, ""label_smoothing"": 0.1, ""loss_preset"": ""all"", ""lr"": 3e-05, ""max_restarts"": 0, ""n_layers"": 3, ""n_seeds"": 3, ""patience"": 10, ""weight_decay"": 0.01, ""word_weighted"": false}"
ADASTT,10,0.1347,"{""colsample_bytree"": 1.0, ""learning_rate"": 0.03, ""max_depth"": 8, ""min_child_weight"": 16.0, ""n_estimators"": 1000, ""objective"": ""cross_entropy"", ""reg_lambda"": 30.0, ""subsample"": 1.0}"
ROVER (confidence-weighted),10,0.1463,"{""alpha"": 0.0, ""conf_source"": ""blend"", ""conf_temp"": 2.0, ""null_penalty"": 0.5, ""weight_by_conf"": true, ""weighting"": ""log_odds""}"
CN-MBR,10,0.1490,"{""candidates"": ""members+consensus"", ""conf_source"": ""probe"", ""decode"": ""slot"", ""null_prior"": 0.2, ""prior"": ""inverse_wer"", ""temperature"": 10.0, ""use_conf"": false}"
MLP-pool (hard CE),10,0.1320,"{""batch_size"": 64, ""d_hidden"": 1024, ""dropout"": 0.05, ""label_smoothing"": 0.1, ""loss_preset"": ""hard_ce_only"", ""lr"": 0.001, ""max_restarts"": 0, ""n_layers"": 3, ""n_seeds"": 3, ""patience"": 5, ""weight_decay"": 0.01, ""word_weighted"": false}"


## 4. 5×2 cross-validation

Five repetitions, each splitting the evaluation clips into two halves with its own seed. Every model is trained on
one half and tested on the other, then the other way round: ten folds, the same for every model.

In [8]:
exp.cross_validate();

  hub: runlog main_earnings22 — 35 of 35 (seed, group) result(s) available
  runlog HIT  cv5x2 seed 123456 — 7/7 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 17 — 7/7 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 99 — 7/7 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 54321 — 7/7 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 8675309 — 7/7 group(s) cached, nothing refitted
5 repetitions x 2 folds x 15 arms


## 5. Main results (Table 3)

In [9]:
exp.results()

Table 3 — main results, Earnings-22
mean ± sd over 10 folds; WER and uWER (mean utterance WER) as fractions, GC (share of the gap between the best single expert and the oracle closed) and Acc (selection accuracy) in %; dist = share of clips routed to Cohere-Transcribe / Kyutai-STT-2.6b / Parakeet-TDT-0.6b-v3
  [expert]
                                 arm             WER            uWER        GC@0      Acc@0           dist
                   Cohere-Transcribe 0.1388 ± 0.0032 0.2168 ± 0.0129 -28.8 ± 7.5 64.1 ± 0.7 1.00-0.00-0.00
                     Kyutai-STT-2.6b 0.1279 ± 0.0024 0.1893 ± 0.0069   0.0 ± 0.0 68.0 ± 0.2 0.00-1.00-0.00
                Parakeet-TDT-0.6b-v3 0.1374 ± 0.0018 0.2181 ± 0.0037 -25.1 ± 4.5 60.8 ± 0.6 0.00-0.00-1.00
Best single expert (Kyutai-STT-2.6b) 0.1279 ± 0.0024 0.1893 ± 0.0069   0.0 ± 0.0 68.0 ± 0.2 0.00-1.00-0.00
  [reference]
   arm             WER            uWER        GC@0      Acc@0           dist
Random 0.1351 ± 0.0032 0.2109 ± 0.0115 -19.0 ± 6.8 64.2 ± 0.4 0.33-0.33-0.34
  [fusion]
                        arm             WER            uWER        GC@0 Acc@0 dist
ROVER (confidence-weighted) 0.1360 ± 0.0036 0.2179 ± 0.0086 -21.3 ± 6.1     —    —
                     CN-MBR 0.1406 ± 0.0040 0.2477 ± 0.0141 -33.3 ± 8.3     —    —
  [pooled]
     arm             WER            uWER       GC@0      Acc@0           dist
MLP-pool 0.1214 ± 0.0036 0.1933 ± 0.0138 17.4 ± 6.5 73.1 ± 0.6 0.67-0.18-0.15
  ADASTT 0.1282 ± 0.0033 0.2094 ± 0.0125 -0.6 ± 6.4 68.9 ± 0.8 0.90-0.03-0.08
  [ours]
    arm             WER            uWER       GC@0      Acc@0           dist
HIT-ASR 0.1198 ± 0.0027 0.1904 ± 0.0061 21.4 ± 3.8 73.2 ± 0.7 0.73-0.12-0.15
  [oracle]
                      arm             WER            uWER        GC@0       Acc@0           dist
                   Oracle 0.0900 ± 0.0018 0.1276 ± 0.0042 100.0 ± 0.0 100.0 ± 0.0 0.64-0.21-0.15
Oracle (1-word tolerance) 0.0990 ± 0.0018 0.1444 ± 0.0045  76.2 ± 0.8  83.8 ± 0.2 0.08-0.84-0.08
Oracle (2-word tolerance) 0.1078 ± 0.0020 0.1596 ± 0.0054  52.9 ± 1.9  75.8 ± 0.3 0.03-0.92-0.05

In [10]:
exp.routing()

router,Cohere-Transcribe,Kyutai-STT-2.6b,Parakeet-TDT-0.6b-v3,entropy,switch rate
Random,33.0,33.3,33.7,1.585,66.7
MLP-pool,66.7,18.2,15.1,1.249,81.8
ADASTT,89.8,2.6,7.6,0.559,97.4
HIT-ASR,73.5,12.0,14.6,1.099,88.0
Oracle,64.1,20.9,15.0,1.293,79.1
Oracle (1-word tolerance),7.6,84.3,8.1,0.785,15.7
Oracle (2-word tolerance),3.3,92.2,4.6,0.473,7.8


## 6. Statistical test (Table 6)

HIT-ASR against every other system on the same ten folds, with a custom statistical test. For full details, please
see the paper.

In [11]:
exp.tests()

Table 6 — HIT-ASR against every system, Earnings-22
on the same 10 folds; effect in the metric's own units
anchor: HIT-ASR; positive effect = anchor better
  [corpus WER]
                 comparator  effect       t      p
          Cohere-Transcribe +0.0190   5.519 0.0027
            Kyutai-STT-2.6b +0.0081   5.265 0.0033
       Parakeet-TDT-0.6b-v3 +0.0176   6.187 0.0016
         Best single expert +0.0081   5.265 0.0033
                     Random +0.0153   5.008 0.0041
ROVER (confidence-weighted) +0.0162   7.948 0.0005
                     CN-MBR +0.0207   5.388 0.0030
                   MLP-pool +0.0015   2.748 0.0404
                     ADASTT +0.0083   3.564 0.0161
                     Oracle -0.0299 -10.746 0.0001
  Oracle (1-word tolerance) -0.0208  -7.391 0.0007
  Oracle (2-word tolerance) -0.0120  -4.160 0.0088
  [mean utterance WER]
                 comparator  effect      t      p
          Cohere-Transcribe +0.0264  2.694 0.0431
            Kyutai-STT-2.6b -0.0011  0.198 0.8508
       Parakeet-TDT-0.6b-v3 +0.0276  2.074 0.0928
         Best single expert -0.0011  0.198 0.8508
                     Random +0.0205  2.188 0.0803
ROVER (confidence-weighted) +0.0275  5.167 0.0036
                     CN-MBR +0.0572  5.911 0.0020
                   MLP-pool +0.0029  2.367 0.0642
                     ADASTT +0.0190  2.394 0.0621
                     Oracle -0.0628 -7.080 0.0009
  Oracle (1-word tolerance) -0.0460 -5.021 0.0040
  Oracle (2-word tolerance) -0.0308 -3.047 0.0285
Custom statistical test — for full details, please see the paper.

## 7. Record

The configuration, the selected hyperparameters and every table above, in `main_earnings22.json`.

In [12]:
exp.save();

wrote main_earnings22.json
